Step-by-Step Lab Execution
Step 1: Create the Source Feedback Table (Silver Layer)
Execute the script below in Databricks SQL Editor or Notebook to establish a realistic baseline table containing raw qualitative feedback from Danantara's operations:

In [0]:
CREATE OR REPLACE TABLE training_medallion.`02_silver`.silver_danantara_qualitative_feedback (
  feedback_id STRING,
  business_unit STRING,        -- e.g., 'Sovereign Mining Fund', 'Macro Quant Strategy', 'Direct Infra PE'
  submitter_role STRING,       -- e.g., 'Internal Compliance Auditor', 'Department Head'
  raw_feedback_text STRING,
  submitted_timestamp TIMESTAMP
) USING DELTA;

INSERT INTO training_medallion.`02_silver`.silver_danantara_qualitative_feedback VALUES
  -- Risk Management Scenario 1: Infrastructure Fund Delays & Governance
  ('FDB-RSK-301', 'Direct Infra PE', 'Senior Risk Controller',
   'Field audit on the East Java Port concession reveals the consortium partner redirected capital expenditure to an unapproved subsidiary without board sign-off. Independent engineer reports a 5-month timeline drift. Potential contract penalty is estimated at USD 14M if commercial operation date is breached by Q1.', current_timestamp()),

  -- Risk Management Scenario 2: Strategic Commodity Liquidity
  ('FDB-RSK-302', 'Sovereign Mining Fund', 'Treasury Risk Specialist',
   'High foreign exchange volatility and falling nickel margins have compressed our counterparty short-term debt servicing coverage ratio (DSCR) below 1.1x. The hedging bank issued a preliminary covenant warning requiring additional cash collateral within 14 business days.', current_timestamp()),

  -- Talent Management Scenario 1: Key Investment Lead Poaching
  ('FDB-TLN-401', 'Macro Quant Strategy', 'Managing Director - Fixed Income',
   'Exit interview notes for the Associate Portfolio Lead indicate substantial career progression fatigue. The individual felt compensation reviews lacked benchmark transparency against international sovereign funds in Singapore. A rival regional fund offered a 35% sign-on bonus and remote flexibility.', current_timestamp()),

  -- Talent Management Scenario 2: Leadership Burnout & Team Friction
  ('FDB-TLN-402', 'Renewable Energy Deal Team', 'Senior HR Business Partner',
   'Quarterly pulse survey comments show severe operational burnout within the offshore wind transaction team. Team members worked 75+ hours weekly for six straight weeks due to repetitive deck revisions. Middle managers cite lack of clear prioritization from the steering committee.', current_timestamp());

num_affected_rows,num_inserted_rows
4,4


Step 2: Validate In-Database Structuring with Schema Enforcement
Before writing back into a permanent Delta table, test the LLM extraction logic. We use Few-Shot System Prompting and explicit JSON contract enforcement inside ai_query() combined with failOnError => false:

In [0]:
SELECT 
  feedback_id,
  business_unit,
  ai_query(
    'system.ai.meta-llama-3-1-8b-instruct',
    CONCAT(
      'Task: Read-only enterprise risk and talent intelligence parser for Danantara Asset Management. ',
      'Convert the provided text into a strict, raw JSON object. Do not output markdown backticks, greetings, or text before/after the JSON. ',
      'JSON Format Contract: {',
        '"domain": "INVESTMENT_RISK" or "TALENT_MANAGEMENT", ',
        '"category": "COVENANT_BREACH" | "LIQUIDITY_PRESSURE" | "KEY_PERSON_ATTRITION" | "LEADERSHIP_FRICTION", ',
        '"severity_tier": "CRITICAL" | "MODERATE" | "LOW", ',
        '"root_cause_tags": ["tag1", "tag2"], ',
        '"financial_or_operational_impact": "summary of financial or operational impact", ',
        '"actionable_recommendation": {',
          '"assigned_unit": "CRO_OFFICE" | "CHRO_OFFICE" | "IC_COMMITTEE" | "DIRECT_LEAD", ',
          '"timeline_days": integer, ',
          '"mitigation_step": "concise next action"',
        '}',
      '}. Text to parse: "', raw_feedback_text, '"'
    ),
    modelParameters => named_struct('temperature', 0.0),
    failOnError => false
  ) AS structured_ai_output
FROM training_medallion.`02_silver`.silver_danantara_qualitative_feedback;

feedback_id,business_unit,structured_ai_output
FDB-RSK-301,Direct Infra PE,"List({ ""domain"": ""INVESTMENT_RISK"", ""category"": ""LIQUIDITY_PRESSURE"", ""severity_tier"": ""CRITICAL"", ""root_cause_tags"": [""unapproved subsidiary"", ""lack of board sign-off""], ""financial_or_operational_impact"": ""Potential contract penalty of USD 14M if commercial operation date is breached by Q1"", ""actionable_recommendation"": { ""assigned_unit"": ""CRO_OFFICE"", ""timeline_days"": 150, ""mitigation_step"": ""Re-evaluate capital expenditure and ensure board sign-off for all future projects"" } }, null)"
FDB-RSK-302,Sovereign Mining Fund,"List({ ""domain"": ""INVESTMENT_RISK"", ""category"": ""LIQUIDITY_PRESSURE"", ""severity_tier"": ""MODERATE"", ""root_cause_tags"": [""foreign_exchange_volatility"", ""falling_nickel_margins""], ""financial_or_operational_impact"": ""Compressed counterparty short-term debt servicing coverage ratio (DSCR) below 1.1x, requiring additional cash collateral"", ""actionable_recommendation"": { ""assigned_unit"": ""CRO_OFFICE"", ""timeline_days"": 14, ""mitigation_step"": ""Provide additional cash collateral"" } }, null)"
FDB-TLN-401,Macro Quant Strategy,"List({ ""domain"": ""TALENT_MANAGEMENT"", ""category"": ""KEY_PERSON_ATTRITION"", ""severity_tier"": ""MODERATE"", ""root_cause_tags"": [""career_progression_fatigue"", ""compensation_transparency""], ""financial_or_operational_impact"": ""Potential loss of key talent due to perceived lack of career progression and competitive compensation."", ""actionable_recommendation"": { ""assigned_unit"": ""CHRO_OFFICE"", ""timeline_days"": 30, ""mitigation_step"": ""Conduct a market benchmarking study to inform compensation reviews and provide transparent career progression opportunities."" } }, null)"
FDB-TLN-402,Renewable Energy Deal Team,"List({ ""domain"": ""TALENT_MANAGEMENT"", ""category"": ""KEY_PERSON_ATTRITION"", ""severity_tier"": ""CRITICAL"", ""root_cause_tags"": [""operational_burnout"", ""lack_of_clear_prioritization""], ""financial_or_operational_impact"": ""Severe operational burnout within the offshore wind transaction team due to repetitive deck revisions, leading to potential delays and increased costs."", ""actionable_recommendation"": { ""assigned_unit"": ""CHRO_OFFICE"", ""timeline_days"": 30, ""mitigation_step"": ""Develop and implement clear prioritization guidelines for the steering committee to ensure efficient use of team resources."" } }, null)"


Step 3: Materialize Enriched JSON into Gold Layer with Typed Extraction
Create the production Gold table. In this step, we:

Extract the JSON substring cleanly using multiline regex (?s)(\\{.*\\}).

Parse specific high-cardinality fields into native SQL columns using get_json_object() and from_json().

Preserve the full JSON string for complex ad-hoc analytics.

In [0]:
CREATE OR REPLACE TABLE training_medallion.`03_gold`.gold_danantara_structured_feedback AS
WITH raw_batch AS (
  SELECT 
    feedback_id,
    business_unit,
    submitter_role,
    raw_feedback_text,
    submitted_timestamp,
    ai_query(
      'system.ai.meta-llama-3-1-8b-instruct',
      CONCAT(
        'Task: Read-only enterprise risk and talent intelligence parser for Danantara Asset Management. ',
        'Convert the text into a strict raw JSON object without markdown fences or comments. ',
        'JSON Schema: {',
          '"domain": "INVESTMENT_RISK" or "TALENT_MANAGEMENT", ',
          '"category": "COVENANT_BREACH" | "LIQUIDITY_PRESSURE" | "KEY_PERSON_ATTRITION" | "LEADERSHIP_FRICTION", ',
          '"severity_tier": "CRITICAL" | "MODERATE" | "LOW", ',
          '"root_cause_tags": ["tag1", "tag2"], ',
          '"financial_or_operational_impact": "impact synopsis", ',
          '"actionable_recommendation": {',
            '"assigned_unit": "CRO_OFFICE" | "CHRO_OFFICE" | "IC_COMMITTEE" | "DIRECT_LEAD", ',
            '"timeline_days": integer, ',
            '"mitigation_step": "mitigation step"',
          '}',
        '}. Input: "', raw_feedback_text, '"'
      ),
      modelParameters => named_struct('temperature', 0.0),
      failOnError => false
    ) AS ai_struct
  FROM training_medallion.`02_silver`.silver_danantara_qualitative_feedback
),
cleaned_stage AS (
  SELECT 
    feedback_id,
    business_unit,
    submitter_role,
    raw_feedback_text,
    submitted_timestamp,
    ai_struct.result AS raw_result,
    ai_struct.errorMessage AS api_error,
    regexp_extract(ai_struct.result, '(?s)(\\{.*\\})', 1) AS valid_json_str
  FROM raw_batch
)
SELECT 
  feedback_id,
  business_unit,
  submitter_role,
  
  -- Top-level categorized attributes
  get_json_object(valid_json_str, '$.domain') AS feedback_domain,
  get_json_object(valid_json_str, '$.category') AS operational_category,
  get_json_object(valid_json_str, '$.severity_tier') AS severity_tier,
  
  -- Impact and root causes
  get_json_object(valid_json_str, '$.financial_or_operational_impact') AS impact_summary,
  from_json(get_json_object(valid_json_str, '$.root_cause_tags'), 'ARRAY<STRING>') AS root_cause_tags,
  
  -- Nested structured mitigation recommendations
  get_json_object(valid_json_str, '$.actionable_recommendation.assigned_unit') AS action_assigned_unit,
  CAST(get_json_object(valid_json_str, '$.actionable_recommendation.timeline_days') AS INT) AS action_timeline_days,
  get_json_object(valid_json_str, '$.actionable_recommendation.mitigation_step') AS action_mitigation_step,
  
  -- Preserved complete JSON string and audit flags
  valid_json_str AS parsed_json_payload,
  CASE 
    WHEN api_error IS NOT NULL THEN 'API_FAILURE'
    WHEN valid_json_str = '' OR get_json_object(valid_json_str, '$.severity_tier') IS NULL THEN 'INVALID_JSON_PAYLOAD'
    ELSE 'SUCCESS'
  END AS pipeline_status,
  submitted_timestamp
FROM cleaned_stage;

num_affected_rows,num_inserted_rows


Step 4: Verification & Operational Querying
Run the following analytical queries to verify how downstream teams interact with the structured output.

Query A: CRO Office – Immediate Portfolio Escalation (Risk Pillar)
Identify all critical risks requiring resolution within 14 days:

In [0]:
SELECT 
  feedback_id,
  business_unit,
  operational_category,
  severity_tier,
  impact_summary,
  action_assigned_unit,
  action_timeline_days,
  action_mitigation_step
FROM training_medallion.`03_gold`.gold_danantara_structured_feedback
WHERE feedback_domain = 'INVESTMENT_RISK'
  AND severity_tier = 'CRITICAL'
  AND action_timeline_days <= 14;

feedback_id,business_unit,operational_category,severity_tier,impact_summary,action_assigned_unit,action_timeline_days,action_mitigation_step
FDB-RSK-302,Sovereign Mining Fund,LIQUIDITY_PRESSURE,CRITICAL,"Compressed counterparty short-term debt servicing coverage ratio (DSCR) below 1.1x, requiring additional cash collateral",CRO_OFFICE,14,Provide additional cash collateral


Query B: CHRO Office – Flight Risk & Burnout Heatmap (Talent Pillar)
Audit talent friction tags and responsible units across departments:


In [0]:
SELECT 
  business_unit,
  operational_category,
  severity_tier,
  root_cause_tags,
  action_assigned_unit,
  action_mitigation_step
FROM training_medallion.`03_gold`.gold_danantara_structured_feedback
WHERE feedback_domain = 'TALENT_MANAGEMENT'
ORDER BY severity_tier DESC;

business_unit,operational_category,severity_tier,root_cause_tags,action_assigned_unit,action_mitigation_step
Macro Quant Strategy,KEY_PERSON_ATTRITION,MODERATE,"List(career progression fatigue, lack of benchmark transparency, poor compensation offer)",CHRO_OFFICE,Review compensation packages to ensure competitiveness with industry benchmarks and offer remote work flexibility to attract and retain top talent
